In [ ]:
!pip install sentence-transformers faiss-cpu scikit-fuzzy numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pickle, time, json
import numpy as np
import faiss
import skfuzzy as fuzz
from sentence_transformers import SentenceTransformer
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

BASE = '/content/drive/MyDrive/semantic-search-system'

index         = faiss.read_index(f'{BASE}/models/faiss.index')
with open(f'{BASE}/models/cluster_model.pkl','rb') as f:
    cluster_model = pickle.load(f)
with open(f'{BASE}/data/processed/clean_corpus.pkl','rb') as f:
    corpus = pickle.load(f)

model = SentenceTransformer('BAAI/bge-base-en-v1.5')
print(f'FAISS : {index.ntotal} vectors | Corpus: {len(corpus["texts"])} docs')

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional
import numpy as np
import faiss
import skfuzzy as fuzz
import time

# ── CacheEntry dataclass ──────────────────────────────────────
@dataclass
class CacheEntry:
    query          : str
    query_embedding: np.ndarray
    result         : Any
    cluster_id     : int
    timestamp      : float = field(default_factory=time.time)
    hit_count      : int   = 0

# ── SemanticCache class ───────────────────────────────────────
class SemanticCache:
    def __init__(self, thr=0.85):
        self.thr = thr
        self._c  = {}
        self._lu = 0
        self._hi = 0

    def lookup(self, emb, cid):
        self._lu += 1
        bs, be = -1.0, None
        for e in self._c.get(cid, []):
            a = emb.flatten().astype(np.float64)
            b = e.query_embedding.flatten().astype(np.float64)
            d = np.linalg.norm(a) * np.linalg.norm(b)
            s = float(np.dot(a, b) / d) if d > 1e-9 else 0.0
            if s > bs:
                bs, be = s, e
        if be and bs >= self.thr:
            self._hi += 1
            be.hit_count += 1
            return {
                'hit'             : True,
                'matched_query'   : be.query,
                'similarity_score': bs,
                'result'          : be.result,
                'cluster_id'      : cid
            }
        return None

    def store(self, q, emb, res, cid):
        self._c.setdefault(cid, []).append(
            CacheEntry(q, emb.copy(), res, cid))

    def stats(self):
        total = sum(len(v) for v in self._c.values())
        return {
            'total_entries' : total,
            'total_lookups' : self._lu,
            'total_hits'    : self._hi,
            'hit_rate'      : round(self._hi / self._lu if self._lu else 0, 4),
            'threshold'     : self.thr
        }

    def reset(self):
        self._c  = {}
        self._lu = 0
        self._hi = 0

cache = SemanticCache(thr=0.85)

# ── Helper functions ──────────────────────────────────────────
def embed_query(text):
    return model.encode(
        [text], normalize_embeddings=True
    )[0].astype(np.float32)


def get_cluster(q_emb):
    """Fixed: unpack all 6 return values from cmeans_predict"""
    q_nd = cluster_model['reducer_nd'].transform(q_emb.reshape(1, -1))

    # cmeans_predict returns 6 values: u, u0, d, jm, p, fpc
    mem, u0, d, jm, p, fpc = fuzz.cluster.cmeans_predict(
        q_nd.T,
        cluster_model['cntr'],
        m=2.0,
        error=0.005,
        maxiter=1000
    )
    return int(np.argmax(mem[:, 0]))


def post_query(query_text, top_k=5):
    t0    = time.time()
    q_emb = embed_query(query_text)
    cid   = get_cluster(q_emb)

    cached = cache.lookup(q_emb, cid)
    if cached:
        return {
            'query'           : query_text,
            'cache_hit'       : True,
            'matched_query'   : cached['matched_query'],
            'similarity_score': round(cached['similarity_score'], 4),
            'dominant_cluster': cid,
            'result'          : cached['result'],
            'latency_ms'      : round((time.time() - t0) * 1000, 2)
        }

    sc, ix = index.search(q_emb.reshape(1, -1), k=top_k)
    results = [
        {
            'doc_id'      : int(i),
            'score'       : round(float(s), 4),
            'category'    : corpus['categories'][i],
            'text_preview': corpus['texts'][i][:200]
        }
        for i, s in zip(ix[0], sc[0])
    ]
    cache.store(query_text, q_emb, results, cid)

    return {
        'query'           : query_text,
        'cache_hit'       : False,
        'matched_query'   : None,
        'similarity_score': None,
        'dominant_cluster': cid,
        'result'          : results,
        'latency_ms'      : round((time.time() - t0) * 1000, 2)
    }

print('All API functions ready')

In [ ]:
test_queries = [
    'symptoms of the common cold and fever',
    'how does the space shuttle work',
    'graphics card not detected in device manager',
    'baseball pitcher statistics ERA',
    'cold symptoms and fever treatment',        # near-duplicate → should HIT
    'space shuttle propulsion and launch',      # near-duplicate → should HIT
]

print('='*65)
for q in test_queries:
    r      = post_query(q)
    status = 'HIT ' if r['cache_hit'] else 'MISS'
    print(f'{status}  cluster={r["dominant_cluster"]:2d}  {r["latency_ms"]:7.1f}ms  {q[:50]}')
    if r['cache_hit']:
        print(f'         matched: {r["matched_query"][:50]}')
        print(f'         score  : {r["similarity_score"]:.4f}')
print('='*65)

In [ ]:
r = post_query('encryption and cryptography algorithms')
clean = {k:v for k,v in r.items() if k != 'result'}
clean['result'] = [{'doc_id':d['doc_id'],'score':d['score'],'category':d['category']}
                   for d in r['result']]
print(json.dumps(clean, indent=2))

In [ ]:
print('GET /cache/stats:')
print(json.dumps(cache.stats(), indent=2))

In [ ]:
cache.reset()
print('DELETE /cache:')
print(json.dumps(cache.stats(), indent=2))

In [ ]:
print('''
The FastAPI server (src/api/main.py) cannot run inside Colab directly.
To run the server, either:

OPTION 1 — Run locally on your computer:
    uvicorn src.api.main:app --reload --port 8000

OPTION 2 — Run in Colab using ngrok (public URL):
    !pip install fastapi uvicorn pyngrok nest-asyncio

Then run the next cell.
''')